In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!git clone https://github.com/agdaniel10/OCSI.git

Cloning into 'OCSI'...
remote: Enumerating objects: 241, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 241 (delta 78), reused 217 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (241/241), 20.79 MiB | 30.24 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [3]:
%cd /kaggle/working/OCSI/src 

/kaggle/working/OCSI/src


In [4]:
!pip install -e ".[perception,yaml]"

Obtaining file:///kaggle/working/OCSI/src
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 2.9 MB/s eta 0:00:00
  Building editable for ocsi (pyproject.toml) ... done
  Created wheel for ocsi: filename=ocsi-0.1.0-0.editable-py3-none-any.whl size=3052 sha256=515e472e8cc1fa0d02713131dee6ccf4d2eba07a181e7e007251e7831cb0ea89
  Stored in directory: /tmp/pip-ephem-wheel-cache-ybp4u6xr/wheels/53/a1/7b/7e3156220194f7ea7c8eed0836c5b9abab1cf160a5d2dc5ec1
Successfully built ocsi


In [6]:
base_dir = "/kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train"

In [7]:
seq_dir = "/kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train/MOT17-02-FRCNN"

In [8]:
import inspect
from ocsi.experiments.mot17_tracking import run_mot17_sequence
print(inspect.signature(run_mot17_sequence))

(seq_dir: 'str', cache_dir: 'str', output_dir: 'str', stages: 'Sequence[str]' = ('baseline', 'memory'), cfg: 'Optional[OCSIConfig]' = None, limit: 'Optional[int]' = None, rebuild_cache: 'bool' = False, detection_source: 'str' = 'yolo', det_conf_threshold: 'Optional[float]' = None, seed: 'Optional[int]' = None, reactivation_app_gate: 'Optional[float]' = None) -> 'Dict'


In [9]:
from ocsi.experiments.mot17_tracking import run_mot17_sequence

payload = run_mot17_sequence(
    seq_dir="/kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train/MOT17-09-FRCNN",
    cache_dir="/kaggle/working/ocsi_cache/MOT17-09-FRCNN-yolov8s-conf015",
    output_dir="/kaggle/working/ocsi_outputs_gate_test/MOT17-09-FRCNN",
    stages=("baseline", "memory", "feedback"),
    detection_source="public",
    det_conf_threshold=0.30,
    rebuild_cache=False,
    reactivation_app_gate=0.85,
)

for r in payload["results"]:
    print(r["stage"], r["summary"])

diag = payload["embedding_diagnostics"]
print("\nembedding diagnostics")
print("  same-id cosine:", diag["same_id_proto_cosine"])
print("  diff-id cosine:", diag["different_id_proto_cosine"])
print("  separation margin:", diag["separation_margin"])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s] 


baseline MOTA= 0.542  IDF1= 0.558  MOTP=0.916  IDsw=  24  FP=    9  FN= 2408  P=0.997 R=0.548  MT=7 PT=16 ML=3
memory MOTA= 0.543  IDF1= 0.547  MOTP=0.916  IDsw=  22  FP=    9  FN= 2400  P=0.997 R=0.549  MT=7 PT=16 ML=3
feedback MOTA= 0.543  IDF1= 0.547  MOTP=0.916  IDsw=  22  FP=    9  FN= 2400  P=0.997 R=0.549  MT=7 PT=16 ML=3

embedding diagnostics
  same-id cosine: 0.9178695004724735
  diff-id cosine: 0.8020321155382775
  separation margin: 0.11583738493419604


In [10]:
import os
from ocsi.experiments.mot17_tracking import run_mot17_sequence

base_dir = "/kaggle/input/datasets/wenhoujinjust/mot-17/MOT17/train"
sequences = sorted([d for d in os.listdir(base_dir) if d.endswith("-FRCNN")])
print(sequences)

all_results = []
for seq in sequences:
    seq_dir = os.path.join(base_dir, seq)
    cache_dir = f"/kaggle/working/ocsi_cache/{seq}-yolov8s-conf015"  # reuse existing cache
    output_dir = f"/kaggle/working/ocsi_outputs_fixed_gate/{seq}"

    payload = run_mot17_sequence(
        seq_dir=seq_dir,
        cache_dir=cache_dir,
        output_dir=output_dir,
        stages=("baseline", "memory", "feedback"),
        detection_source="public",
        det_conf_threshold=0.30,
        rebuild_cache=False,          # cache already built from the earlier run — reuse it
        reactivation_app_gate=0.85,   # the fix
    )
    all_results.append(payload)

    for r in payload["results"]:
        print(seq, r["stage"], r["summary"])

    diag = payload["embedding_diagnostics"]
    print(f"  {seq} same-id: {diag['same_id_proto_cosine']:.3f}  "
          f"diff-id: {diag['different_id_proto_cosine']:.3f}  "
          f"margin: {diag['separation_margin']:.3f}")
    print("-" * 60)

['MOT17-02-FRCNN', 'MOT17-04-FRCNN', 'MOT17-05-FRCNN', 'MOT17-09-FRCNN', 'MOT17-10-FRCNN', 'MOT17-11-FRCNN', 'MOT17-13-FRCNN']
MOT17-02-FRCNN baseline MOTA= 0.256  IDF1= 0.357  MOTP=0.887  IDsw= 112  FP= 1327  FN=12385  P=0.824 R=0.333  MT=7 PT=23 ML=32
MOT17-02-FRCNN memory MOTA= 0.258  IDF1= 0.360  MOTP=0.886  IDsw=  88  FP= 1333  FN=12366  P=0.823 R=0.334  MT=7 PT=24 ML=31
MOT17-02-FRCNN feedback MOTA= 0.258  IDF1= 0.360  MOTP=0.886  IDsw=  88  FP= 1333  FN=12366  P=0.823 R=0.334  MT=7 PT=24 ML=31
  MOT17-02-FRCNN same-id: 0.909  diff-id: 0.727  margin: 0.182
------------------------------------------------------------
MOT17-04-FRCNN baseline MOTA= 0.506  IDF1= 0.599  MOTP=0.898  IDsw=  43  FP= 1810  FN=21624  P=0.935 R=0.545  MT=17 PT=42 ML=24
MOT17-04-FRCNN memory MOTA= 0.506  IDF1= 0.599  MOTP=0.898  IDsw=  43  FP= 1810  FN=21622  P=0.935 R=0.545  MT=17 PT=42 ML=24
MOT17-04-FRCNN feedback MOTA= 0.506  IDF1= 0.599  MOTP=0.898  IDsw=  43  FP= 1810  FN=21622  P=0.935 R=0.545  MT=17 